In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np


In [20]:
TRAINING_DATA = "training_data_clean.csv"
from google.colab import files


def prep_data():
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)

    # rename
    return df


In [47]:
import numpy as np
import re
from sklearn.compose import ColumnTransformer
 # TODO: can use KNN to find the imputation point instead, in-person imputation
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import FunctionTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer



# divide columns by type
text_cols = [
    'best_tasks_free',
    'subopt_tasks_free',
    'verify_method_free',
    ]


ord_cols = [
    'acad_tasks_rating',
    'subopt_freq_rating',
    'evidence_freq_rating',
    'verify_freq_rating',
    ]

ord_categories = [
                '1 — Not at all likely',
                '2 — Unlikely',
                '3 — Neutral / Unsure',
                '4 — Likely',
                '5 — Very likely'
                ]

cat_cols = [
    'best_tasks_select',
    'subopt_tasks_select',
    ]

def preprocess_text():
    return make_pipeline(
        SimpleImputer(strategy="constant", fill_value ="", missing_values=np.nan)
        # text embedding -> always use llm, tokenize and get embedding vector, using hugging face library (Ex. BERT)
        # return torch tensor, embedding is one of the properties

        # TFIDF manually (classical)
    )


def preprocess_ord():

    return make_pipeline(
        OrdinalEncoder(categories= [ord_categories] * len(ord_cols), handle_unknown='use_encoded_value', unknown_value=np.nan),
        IterativeImputer(
            max_iter=20,
            sample_posterior=True
        )

    )

def preprocess_cat():
    pass
    # TODO: research how to preprocess these

def create_preprocessor():
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem
    transformers = [
        # ("preprocessing_pipeline_for_free_response_features", preprocess_text(), text_cols),
        ("preprocessing_pipeline_for_ordinal_rating_features", preprocess_ord(), ord_cols),
        #  ("preprocessing_pipeline_for_categorical_select_features", preprocess_cat(), cat_cols),
    ]

    return ColumnTransformer(transformers=transformers)


# column_transformer = create_preprocessor()
# column_transformer.set_output(transform="pandas")
# preprocessed_num_cat_features_df = column_transformer.fit_transform(
#     train_df[[NUMERICAL_FEATURE, CATEGORICAL_FEATURE]]
# )

In [48]:

def split_data(df):
    df.columns = [
        'id', 'best_tasks_free', 'acad_tasks_rating', 'best_tasks_select', 'subopt_freq_rating',  'subopt_tasks_select', 'subopt_tasks_free',
        'evidence_freq_rating', 'verify_freq_rating', 'verify_method_free', 'class']
    students = df['id'].unique()
    train_idx, test_idx = train_test_split(students, test_size=0.3, random_state=22)

    X_train_sets = []
    X_test_sets = []

    train_df = df[df['id'].isin(train_idx)]
    test_df = df[df['id'].isin(test_idx)]


    X_train = train_df.drop(columns=['class'])
    y_train = train_df['class']



    X_test = test_df.drop(columns=['class'])
    y_test = test_df['class']

    M = 5
    for i in range(M):

      curr_preprocessor = create_preprocessor()
      curr_X_train = curr_preprocessor.fit_transform(X_train)
      # only tranfrom to prevent data leakage
      curr_X_test = curr_preprocessor.transform(X_test)

      X_train_sets.append(curr_X_train)
      X_test_sets.append(curr_X_test)

    return (X_train_sets, y_train, X_test_sets, y_test)

In [51]:
df = prep_data()

X_train_sets, y_train, X_test_sets, y_test = split_data(df)




Saving training_data_clean.csv to training_data_clean (14).csv
[[2.]
 [3.]
 [2.]
 [4.]
 [3.]
 [0.]
 [2.]
 [0.]
 [0.]
 [3.]]
[[2.]
 [3.]
 [2.]
 [4.]
 [3.]
 [0.]
 [2.]
 [0.]
 [0.]
 [3.]]
[[2.]
 [3.]
 [2.]
 [4.]
 [3.]
 [0.]
 [2.]
 [0.]
 [0.]
 [3.]]
[[2.]
 [3.]
 [2.]
 [4.]
 [3.]
 [0.]
 [2.]
 [0.]
 [0.]
 [3.]]
[[2.]
 [3.]
 [2.]
 [4.]
 [3.]
 [0.]
 [2.]
 [0.]
 [0.]
 [3.]]


/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [1 2 3]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [1 2 3]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [1 2 3]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: UserWarning: Skipping features without any observed values: [1 2 3]. At least one non-missing value is needed for imputation with strategy='mean'.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/impute/_base.py:635: